In [3]:
import os
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator

def generate_compact_interface_1A(elec_file, anode_file):
    # print("🔨 启动紧密版构建 (去除 Na 真空, Gap=1.0A)...")

    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print("❌ 找不到文件！")
        return

    # 1. 读取结构
    elyte = Structure.from_file(elec_file).get_primitive_structure()
    anode = Structure.from_file(anode_file).get_primitive_structure()

    # 2. 切片 (这里我们把真空设小一点，方便处理，但关键是后面的堆叠步骤)
    # min_vacuum_size=0 在 pymatgen 有时会报错，设为 1.0 只要能生成对象即可
    # 我们后面会手动忽略这个真空
    slab_elyte = SlabGenerator(elyte, (0,0,1), min_slab_size=9, min_vacuum_size=1.0, center_slab=True).get_slab()
    slab_anode = SlabGenerator(anode, (0,0,1), min_slab_size=9, min_vacuum_size=1.0, center_slab=True).get_slab()

    # 3. ZSL 匹配 (只匹配 XY 平面，和 Z 轴真空无关)
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2, 
        max_area=400,          
        max_length_tol=0.2,    
        max_angle_tol=0.1
    )

    print(zsl)

    u_vec = slab_elyte.lattice.matrix[0:2, 0:2]
    v_vec = slab_anode.lattice.matrix[0:2, 0:2]

    matches = list(zsl(u_vec, v_vec))
    if not matches:
        print("❌ 未找到匹配。")
        return

    # 4. 筛选最佳原子数 (目标 ~120 电解质原子)
    best_match = None
    best_score = 9999
    target_elyte_atoms = 120 

    for match in matches:
        # 兼容不同版本 pymatgen
        if hasattr(match, "substrate_transformation"):
            sub_m = np.array(match.substrate_transformation)
            film_m = np.array(match.film_transformation)
        elif isinstance(match, dict):
            sub_m = np.array(match.get('sub_matrix'))
            film_m = np.array(match.get('film_matrix'))
        else:
            continue

        n_elyte = len(slab_elyte) * abs(np.linalg.det(sub_m))
        score = abs(n_elyte - target_elyte_atoms)
        
        if score < best_score:
            best_score = score
            best_match = (sub_m, film_m)

    # 5. 执行 XY 扩胞
    m_elyte = np.eye(3); m_elyte[0:2, 0:2] = best_match[0]
    m_anode = np.eye(3); m_anode[0:2, 0:2] = best_match[1]

    super_elyte = slab_elyte.copy(); super_elyte.make_supercell(m_elyte)
    super_anode = slab_anode.copy(); super_anode.make_supercell(m_anode)

    # ================= 自动加厚 Na 层 =================
    current_na_atoms = len(super_anode)
    # 我们希望 Na 层足够厚，形成块体，而不是薄膜
    if current_na_atoms < 40:
        z_scale = int(np.ceil(40 / current_na_atoms))
        print(f"  - ⚡ Na 层原子数 {current_na_atoms} -> 自动加厚 x{z_scale} 倍")
        super_anode.make_supercell([1, 1, z_scale])
    # =================================================

    # 6. 堆叠与去除真空 (关键步骤)
    # ---------------------------------------------------------
    interface_gap = 1.0  # 设为 1 埃
    # ---------------------------------------------------------

    # 计算原子的真实占据厚度 (Real Atomic Thickness)
    # 这一步完全忽略了 SlabGenerator 生成的那个真空层
    z_e = [s.coords[2] for s in super_elyte]
    thick_e = max(z_e) - min(z_e)
    
    z_a = [s.coords[2] for s in super_anode]
    thick_a = max(z_a) - min(z_a)

    print(f"  - 真实物理厚度: Electrolyte={thick_e:.2f} Å, Anode={thick_a:.2f} Å")

    # 新的总高度 = 两个实体厚度 + 1.0 间隙
    # 这样生成的盒子，除了这 1.0 的缝隙，其他地方全是实心的
    total_c = thick_e + thick_a + interface_gap

    # 构建新晶格
    final_matrix = super_elyte.lattice.matrix.copy()
    final_matrix[2, 0] = 0; final_matrix[2, 1] = 0
    final_matrix[2, 2] = total_c

    final_struct = Structure(Lattice(final_matrix), [], [])

    # 6.1 放入 Elyte (底部)
    min_z_e = min(z_e)
    for s in super_elyte:
        new_c = list(s.coords)
        new_c[2] -= min_z_e # 归零
        new_c[2] += 0.5     # 底部稍微抬起 0.5，为了美观，实际上 PBC 下无所谓
        final_struct.append(s.specie, new_c, coords_are_cartesian=True)

    # 6.2 放入 Anode (顶部)
    # 计算 Na 的放置高度：底部缓冲 + 电解质实体厚度 + 1.0 间隙
    # 注意：这里我们紧贴着放，完全不给原来的真空层留位置
    start_z_anode = 0.5 + thick_e + interface_gap
    min_z_a = min(z_a)
    
    vec_a = final_matrix[0]; vec_b = final_matrix[1]

    for s in super_anode:
        u, v = s.frac_coords[0], s.frac_coords[1]
        
        # 坐标变换：把 Na 的原子搬运过来
        old_z = s.coords[2]
        new_z = (old_z - min_z_a) + start_z_anode
        
        pos = u * vec_a + v * vec_b
        pos[2] = new_z
        
        final_struct.append(s.specie, pos, coords_are_cartesian=True)

    # 7. 保存与检查
    final_struct.perturb(0.05) # 微小扰动，打破对称性
    
    save_name = f"Na_Na3SbS4_Data/Interface_structures/Interface_Gap_Compact_{len(final_struct)}atoms.vasp"
    os.makedirs(os.path.dirname(save_name), exist_ok=True)
    final_struct.to(filename=save_name, fmt="poscar")

    print("\n✅ 生成完成！")
    print(f"  - 保存路径: {save_name}")
    print(f"  - 总原子数: {len(final_struct)}")
    print(f"  - 盒子高度 (C轴): {total_c:.2f} Å")
    print(f"  - 界面间距: {interface_gap} Å")
    print(f"  - 状态: 已彻底移除 Na 层自带的真空，实现 {interface_gap} Å 紧密接触。")



In [4]:
actual_elec_path = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
actual_anode_path = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"

generate_compact_interface_1A(actual_elec_path, actual_anode_path)

  - ⚡ Na 层原子数 8 -> 自动加厚 x5 倍
  - 真实物理厚度: Electrolyte=20.41 Å, Anode=85.03 Å

✅ 生成完成！
  - 保存路径: Na_Na3SbS4_Data/Interface_structures/Interface_Gap_Compact_168atoms.vasp
  - 总原子数: 168
  - 盒子高度 (C轴): 106.44 Å
  - 界面间距: 1.0 Å
  - 状态: 已彻底移除 Na 层自带的真空，实现 1.0 Å 紧密接触。


In [11]:
import os
import math
import warnings
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.io.vasp.inputs import Kpoints

# ================= 配置区域 =================

input_dir = "Na_Na3SbS4_Data/Interface_structures"
base_output_dir = "Na_Na3SbS4_Data/Interface_AIMD_runs"
os.makedirs(base_output_dir, exist_ok=True)

# AIMD 参数设置 (保持你的设置不变)
aimd_settings = {
    "IBRION": 0,          
    "NSW": 5000,          
    "POTIM": 2.0,         
    "TEBEG": 800,         
    "TEEND": 800,
    "ISYM": 0,            
    "SMASS": 0,           
    "ISIF": 2,            
    "KBLOCK": 1,          
    "ALGO": "Fast",       
    "PREC": "Normal",     
    "LREAL": "Auto",      
    "NELM": 100,          
    "ISMEAR": -1,         # Fermi Smearing (适合金属界面)
    "SIGMA": 0.1,         
    "ISPIN": 1,           
    "LWAVE": False,       
    "LCHARG": False,      
    "NCORE": 8,           
}

# ================= 处理流程 =================
print(f"📂 读取结构目录: {input_dir} ...")

if not os.path.exists(input_dir):
    print(f"❌ 错误: 找不到目录 {input_dir}")
else:
    files = [f for f in os.listdir(input_dir) if f.endswith(('.cif', '.vasp'))]
    files.sort() 
    print(f"🔍 发现 {len(files)} 个结构文件。")

    for filename in files:
        file_path = os.path.join(input_dir, filename)
        struct_name = os.path.splitext(filename)[0]
        
        try:
            # 1. 加载结构
            structure = Structure.from_file(file_path)
            
            # 2. 扩胞检查
            min_length = 10.0
            lengths = structure.lattice.abc
            scaling_matrix = [max(1, int(math.ceil(min_length / l))) for l in lengths]
            
            if any(x > 1 for x in scaling_matrix):
                print(f"  - {struct_name}: 执行扩胞 {scaling_matrix}")
                structure.make_supercell(scaling_matrix)
            
            # 3. 创建目录
            task_dir = os.path.join(base_output_dir, struct_name)
            os.makedirs(task_dir, exist_ok=True)
            
            # 4. 生成 VASP 输入文件 (修复点在这里!)
            # 我们先创建 Kpoints 对象
            gamma_only = Kpoints.gamma_automatic()
            
            # 然后直接传给 MPRelaxSet
            vis = MPRelaxSet(
                structure, 
                user_incar_settings=aimd_settings,
                user_potcar_functional="PBE",
                user_kpoints_settings=gamma_only  # <--- 直接在这里传入对象
            )
            
            # 写入文件
            vis.write_input(task_dir)
            print(f"✅ 生成成功: {struct_name} (原子数: {structure.num_sites}) -> {task_dir}")
            
        except Exception as e:
            print(f"❌ 跳过 {filename}: {e}")

    print(f"\n🎉 全部完成！请检查 '{base_output_dir}'")

📂 读取结构目录: Na_Na3SbS4_Data/Interface_structures ...
🔍 发现 1 个结构文件。
✅ 生成成功: Interface_Gap1A_Compact_168atoms (原子数: 168) -> Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Gap1A_Compact_168atoms

🎉 全部完成！请检查 'Na_Na3SbS4_Data/Interface_AIMD_runs'


In [13]:
import os
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 配置区域 =================

# 1. 基础路径
work_base = "Na_Na3SbS4_Data/Interface_AIMD_runs" 

# 2. 指定你要跑的那个文件夹名字 (精准打击)
target_folder_name = "Interface_Gap1A_Compact_168atoms"

# 3. VASP 路径
vasp_exe = "/dssg/opt/icelake/linux-centos8-icelake/oneapi-2021.4.0/vasp/vasp.6.3.0/bin/vasp_std"

# ================= 机器与资源 =================

machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=1,
    module_list=["vasp/6.3.0-intel-2021.4.0"], 
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --ntasks=64",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn",
        "#SBATCH --time=96:00:00",      
        f"#SBATCH --job-name={target_folder_name}" # 任务名直接用文件夹名
    ]
)

setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)
command = f"{setup_env} && mpirun -n 64 {vasp_exe}"

# ================= 构建任务 =================

task_list = []
full_path = os.path.join(work_base, target_folder_name)

print(f"🎯 正在定位目标任务: {full_path} ...")

# 检查文件夹是否存在，并且里面有 INCAR
if os.path.isdir(full_path) and os.path.exists(os.path.join(full_path, "INCAR")):
    print(f"  ✅ 找到任务文件夹，准备提交...")
    
    task = Task(
        command=command,
        task_work_path=target_folder_name, # 注意：这里填相对路径(文件夹名)即可，因为 Submission 指定了 work_base
        forward_files=['INCAR', 'POSCAR', 'POTCAR', 'KPOINTS'], 
        backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR', 'XDATCAR', 'CONTCAR'] 
    )
    task_list.append(task)
else:
    print(f"❌ 错误: 找不到文件夹 {full_path} 或其中缺少 INCAR 文件！")

# ================= 提交执行 =================

if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )
    
    submission.run_submission()
    print(f"🚀 任务 {target_folder_name} 已提交！")
    print("📋 请使用 'squeue' 查看状态。")
else:
    print("❌ 提交终止。")

🎯 正在定位目标任务: Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Gap1A_Compact_168atoms ...
  ✅ 找到任务文件夹，准备提交...
2025-12-13 13:53:04,042 - INFO : info:check_all_finished: False
2025-12-13 13:53:04,160 - INFO : job: a09c58dc262e2d6575dc898a4845309a7e2f53d0 submit; job_id is 50844402


In [1]:
!ls

0a8901819275397a17cbeceae0ee7ca914e64552_job_id   main-3.ipynb
0a8901819275397a17cbeceae0ee7ca914e64552.sub	  Na_Na3SbS4_Data
0a8901819275397a17cbeceae0ee7ca914e64552.sub.run  scripts
b53d8cb1575b4627a9d6c67528a3b3feb7f6c6f0.json	  test-interface.ipynb
dpdispatcher.log				  vasp-interface.ipynb
LiGa						  vasp-Na3SbS4.ipynb


In [2]:
import os
import shutil
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 配置区域 =================

# 1. 基础路径
work_base = "Na_Na3SbS4_Data/Interface_AIMD_runs"

# 2. 上一次中断的任务文件夹名 (来源)
prev_folder_name = "Interface_Gap1A_Compact_168atoms"

# 3. 新的任务文件夹名 (自动加上 _continue 后缀)
# 这样能保证文件不冲突，最安全
new_folder_name = f"{prev_folder_name}_continue"

# 4. VASP 路径
vasp_exe = "/dssg/opt/icelake/linux-centos8-icelake/oneapi-2021.4.0/vasp/vasp.6.3.0/bin/vasp_std"

# ================= 1. 自动准备文件 =================

src_path = os.path.join(work_base, prev_folder_name)
dst_path = os.path.join(work_base, new_folder_name)

print(f"🔄 正在准备续跑环境...")
print(f"  - 来源: {src_path}")
print(f"  - 目标: {dst_path}")

# 检查上一步是否有 CONTCAR
if not os.path.exists(os.path.join(src_path, "CONTCAR")):
    raise FileNotFoundError("❌ 上一个任务没有生成 CONTCAR，无法续跑！可能是任务刚开始就挂了。")

# 检查 CONTCAR 是否为空
if os.path.getsize(os.path.join(src_path, "CONTCAR")) == 0:
    raise ValueError("❌ CONTCAR 文件大小为 0，数据损坏，无法续跑。")

# 创建新目录
os.makedirs(dst_path, exist_ok=True)

# 复制文件
try:
    # 关键：CONTCAR -> POSCAR
    shutil.copy(os.path.join(src_path, "CONTCAR"), os.path.join(dst_path, "POSCAR"))
    print("  ✅ 已将 CONTCAR 复制为新的 POSCAR")
    
    # 复制其他输入文件
    for f in ["INCAR", "POTCAR", "KPOINTS"]:
        shutil.copy(os.path.join(src_path, f), os.path.join(dst_path, f))
    print("  ✅ 配置文件 (INCAR/POTCAR/KPOINTS) 已复制")
    
except Exception as e:
    print(f"❌ 文件复制出错: {e}")
    # 中止后续操作
    raise 

# ================= 2. 配置机器资源 =================

machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=1,
    module_list=["vasp/6.3.0-intel-2021.4.0"], 
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --ntasks=64",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn",
        "#SBATCH --time=96:00:00",      
        f"#SBATCH --job-name={new_folder_name}" # 使用新名字
    ]
)

setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)
command = f"{setup_env} && mpirun -n 64 {vasp_exe}"

# ================= 3. 提交任务 =================

task_list = []

# 再次检查新目录下是否有 POSCAR
if os.path.exists(os.path.join(dst_path, "POSCAR")):
    task = Task(
        command=command,
        task_work_path=new_folder_name, # 填新文件夹名
        forward_files=['INCAR', 'POSCAR', 'POTCAR', 'KPOINTS'], 
        backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR', 'XDATCAR', 'CONTCAR'] 
    )
    task_list.append(task)
    
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )
    
    print(f"🚀 正在提交续跑任务: {new_folder_name} ...")
    submission.run_submission()
    print("✅ 提交成功！")
else:
    print("❌ 准备工作似乎失败了，新目录下没有 POSCAR。")

🔄 正在准备续跑环境...
  - 来源: Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Gap1A_Compact_168atoms
  - 目标: Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Gap1A_Compact_168atoms_continue
  ✅ 已将 CONTCAR 复制为新的 POSCAR
  ✅ 配置文件 (INCAR/POTCAR/KPOINTS) 已复制
🚀 正在提交续跑任务: Interface_Gap1A_Compact_168atoms_continue ...
2025-12-14 19:15:52,558 - INFO : info:check_all_finished: False
2025-12-14 19:15:52,636 - INFO : job: 1c129794e991f09703f5bed74f53997621cfc1e5 submit; job_id is 51002177
